# LLM基礎: 大規模言語モデルを基礎から作る

### Google Colab 実習ノートブック

上から順番にセルを実行してください。開始前にGPUランタイムを選択します。


# はじめに：この本の読み方

## この本の目的

本書は、ChatGPTやGPT-4のような大規模言語モデル（LLM, Large Language Model）が**内部でどのように動作しているのか**を、最初から最後までコードを直接実装しながら学ぶ実践的な教材です。

理論だけを学んでも「アテンションが重要である」という言葉の意味は理解できても、実際になぜ、どのように動作するのかという感覚は掴めません。逆に、コードだけを写していくと、原理を理解しないままコピー＆ペーストを繰り返すことになります。本書では、この両方を同時に扱います。

- **理論**: 各章の冒頭で、概念を図や数式なしでも理解できるよう、比喩を用いて説明します。
- **実践**: すべての章に、Google Colabですぐに実行可能なPythonコードが含まれています。PyTorchを使用して、トークナイザー、埋め込み（Embedding）、アテンション、Transformerブロックを一行ずつ直接作り上げ、最終的には小さなGPTモデルを学習させてテキストを生成します。

本書を最後まで進めることで、以下の要素を自力で実装できるようになります。

1. テキストをトークンに分割するBPEトークナイザー
2. 単語をベクトルに変換する埋め込み層（Embedding layer）
3. Self-AttentionとMulti-Head Attention
4. Transformerブロック（LayerNorm, FeedForward, Residual Connection）
5. ミニGPTモデルの全体構造
6. 学習ループと損失関数
7. テキスト生成のサンプリング戦略（top-k, top-p, temperature）
8. Hugging Faceの学習済みモデルの読み込み
9. LoRAを用いた軽量なファインチューニング

## 必要な事前知識

Pythonの文法を知っており、ディープラーニングが初めてであっても問題ありません。ただし、以下の概念を聞いたことがあると、よりスムーズに学習を進められます。

- 行列演算（内積、次元）
- ニューラルネットワークの基本構造（入力層、隠れ層、出力層）
- 勾配降下法（gradient descent）という言葉

これらを知らなくても、各章で必要な分だけ簡潔に説明しながら進めていきます。

## GitHubリポジトリとGoogle Colabの使い方

本書のすべての実践コードとデモスクリプトは、以下のGitHubリポジトリで公開・管理されています。ノートブックファイルを直接ダウンロードしたり、ブラウザで開いたりすることができます。

- **GitHubリポジトリ**: [https://github.com/dirmich/mini_gpt_release](https://github.com/dirmich/mini_gpt_release)

Google Colabは、ブラウザ上で無料でGPUを利用できるPython実行環境です。別途インストールすることなく、以下の手順で開始してください。

1. [colab.research.google.com](https://colab.research.google.com) にアクセスしてログイン
2. 上部メニューの `ファイル > ノートブックをアップロード` を通じて、上記のGitHubリポジトリからダウンロードした `.ipynb` ファイルを読み込む
3. `ランタイム > ランタイムのタイプを変更` で、ハードウェア アクセラレータを **GPU**（T4以上を推奨）に設定
4. 各コードブロックを順番に1つのセルに貼り付け、`Shift + Enter` で実行

本書では、コードブロックが登場するたびに、コメントで次のように表示します。


In [ ]:
# ── Colab セル ──


この注釈が表示されたら、新しいセルに貼り付けて実行するものと考えてください。同じ章内で続くセルは、前のセルで定義した変数や関数を継続して使用するため、**必ず順番に実行**する必要があります。

## モジュール構造の案内

演習コードは、一つの巨大なスクリプトではなく、実際のオープンソースLLMライブラリのように、役割ごとに分割されたモジュールで構成します。

```
mini_gpt/
├── tokenizer.py       # 第2章: BPE トークナイザー
├── embedding.py        # 第3章: トークン埋め込み + 位置エンコーディング
├── attention.py         # 第4章: Self-Attention, Multi-Head Attention
├── transformer_block.py # 第5章: Transformer ブロック
├── model.py             # 第6章: GPT モデル全体の組み立て
├── train.py              # 第7章: 学習ループ
└── generate.py           # 第8章: テキスト生成
```

Colabでは、ファイルを複数作成する代わりに `%%writefile` マジックコマンドを使用して各モジュールをファイルとして保存し、その後のセルで `import` して使用します。これにより、実際のプロジェクトのようにモジュール化された構造を、Colab環境でもそのまま体験することができます。

それでは、始めていきましょう。


# LLMとは何か

## 言語モデルの定義

言語モデル（Language Model）を極めて単純に表現すると、**「次にくる単語を確率的に予測する機械」**です。

> 「今日の夕食に ___ を食べた」

空欄に「ご飯」「チキン」「ラーメン」といった単語が入る確率は高いですが、「自動車」「宇宙船」といった単語が入る確率は低くなります。言語モデルは、膨大なテキストを学習することで、このような確率分布を統計的に習得します。ChatGPTのようなモデルが文章を生成するプロセスも、根本的には「次のトークンを一つずつ予測して繋ぎ合わせていく」という反復作業です。

ここで「大規模（Large）」という言葉が付く理由は、主に2つあります。

1. **パラメータ数が多い**: 数億〜数兆個の学習可能な重み（Weights）を持っています。
2. **学習データが膨大**: インターネット上のテキスト、書籍、コードなど、数千億〜数兆トークンを学習します。

## なぜTransformerなのか

2017年の論文「Attention Is All You Need」で提案された**Transformer**構造は、現在ほぼすべてのLLMの基盤となっています。Transformer以前はRNN（回帰型ニューラルネットワーク）が主に使われていましたが、RNNは文章を一単語ずつ順番に処理する必要があるため速度が遅く、文章が長くなると文頭の情報が失われるという問題がありました。

Transformerは**Attention（アテンション）**メカニズムを利用して、文章内のすべての単語を同時に参照し、各単語が他の単語とどれほど関連しているかを計算します。これにより、以下のメリットが得られます：

- 並列処理が可能になり、学習が高速化する
- 文章が長くても、遠く離れた単語間の関係性を捉えやすい

本書で作るミニGPTも、このTransformer構造をそのまま縮小したバージョンです。

## GPT系モデルの動作原理（一目で理解する）

GPT（Generative Pre-trained Transformer）系のモデルは、以下のパイプラインで動作します。```
テキスト入力
   │
   ▼
[トークナイザー] テキスト → トークンIDシーケンス
   │
   ▼
[エンベディング] トークンID → ベクトル、位置情報の追加
   │
   ▼
[トランスフォーマー・ブロック × N] Self-Attention + FeedForward の繰り返し
   │
   ▼
[出力層] 次のトークンの確率分布を計算
   │
   ▼
[サンプリング] 確率分布から次のトークンを選択
   │
   ▼
生成されたトークンを入力に結合して繰り返す
```

この本の第2章から第8章は、まさにこのパイプラインの各ステップに対応しています。つまり、このダイアグラムが本書全体の地図となります。

## 事前学習とファインチューニング

LLMは通常、2つの段階を経て構築されます。

- **事前学習 (Pre-training)**: 大量のテキストを用いて「次のトークンを予測する」学習のみを繰り返します。この段階が終わると、文法、常識、世界知識をある程度備えたモデルになります。しかし、まだ「質問に答える方法」は知りません。
- **ファインチューニング (Fine-tuning)**: 事前学習済みモデルを特定の目的（対話、指示の実行、コード作成など）に合わせて、少量のデータで追加学習させます。これには、教師あり学習によるファインチューニング (SFT)、人間からのフィードバックに基づく強化学習 (RLHF)、LoRAのような軽量化手法などが含まれます。

本書では、第6〜7章でミニGPTをゼロから事前学習させ、第11章で事前学習済みの実モデルをLoRAを用いて軽量にファインチューニングするまでを扱います。

## 開発環境の準備 (最初の Colab セル)

それでは、最初のコードを実行してみましょう。ColabでGPUが正しく認識されているかを確認し、必要なパッケージをインストールします。


In [ ]:
# ── Colab セル ──
# GPUの確認
import torch
print("PyTorch バージョン:", torch.__version__)
print("CUDA 使用可能:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU 名:", torch.cuda.get_device_name(0))

device = "cuda" if torch.cuda.is_available() else "cpu"
print("使用するデバイス:", device)


In [ ]:
# ── Colab セル ──
# この本全体で使用するパッケージのインストール
!pip install -q torch transformers datasets tiktoken regex peft accelerate


`torch.cuda.is_available()` が `False` と表示される場合は、「ランタイム > ランタイムのタイプを変更」で GPU を T4 に設定しているか確認してください。CPUのみでも本書の実習を進めることは可能ですが、第7章の学習フェーズでは GPU を使用したほうがはるかに高速です。

次の章からは、テキストをモデルが理解できる数値に変換する **トークナイザー (Tokenizer)** を実際に作成していきます。


# トークナイゼーション：テキストを数値へ

## なぜトークンが必要なのか

ニューラルネットワークは数値しか理解できません。「こんにちは」という文字列をそのままモデルに入力することはできず、整数のIDの羅列に変換する必要があります。この変換プロセスを**トークナイゼーション（Tokenization）**と呼び、変換ルールを保持するツールのことを**トークナイザー（Tokenizer）**と呼びます。

最も単純な方法は「単語単位で分割し、各単語に番号を振る」ことです。しかし、この方法には2つの問題があります。

1. 辞書にない単語（新造語、誤字、外国語）を処理できない。
2. 単語数が多すぎると、語彙（vocabulary）が膨大になりすぎる。

逆に「文字単位で分割する」方法は、語彙は小さくなりますが、文章が極端に長くなり、一文字ごとの意味がほとんどないため学習の効率が悪くなります。

## BPE：2つの手法の折衷案

**BPE（Byte Pair Encoding）**は、実際のGPT系モデルの多くで使用されている方式で、頻繁に出現する文字のペアを繰り返し結合しながら語彙を作成します。例えば、"low", "lower", "lowest" という単語が頻繁に登場する場合、`l`+`o` → `lo`、`lo`+`w` → `low` のように、よく隣り合う断片を徐々に大きな単位へと結合していきます。

その結果、頻繁に使われる単語は丸ごと一つのトークンになり、珍しい単語は複数のサブワード（subword）トークンに分割されます。これにより、語彙のサイズを調整しつつ、未知の単語であってもサブワードの組み合わせによって表現することが可能になります。

## BPE 学習アルゴリズム

1. テキストを文字単位で分割する。
2. 最も頻繁に出現する隣接する2つのトークンのペアを見つける。
3. そのペアを一つの新しいトークンとして結合する。
4. 目標とする語彙サイズに達するまで、手順2〜3を繰り返す。

それでは、実際に実装してみましょう。Colabの新しいセルに、以下のコードを順番に入力してください。


In [ ]:
# ── Colab セル ──
# mini_gpt パッケージフォルダの作成
import os
os.makedirs("mini_gpt", exist_ok=True)


In [ ]:
# ── Colab セル ──
%%writefile mini_gpt/tokenizer.py
"""BPE (Byte Pair Encoding) トークナイザー - スクラッチからの実装"""
from collections import Counter, defaultdict


class BPETokenizer:
    def __init__(self):
        self.merges = {}          # (token1, token2) -> 結合された新しいトークン ID
        self.vocab = {}           # トークン ID -> バイトシーケンス
        self.token_to_id = {}     # 文字列 -> トークン ID

    def _get_pair_counts(self, token_ids_list):
        """すべてのシーケンスにおける隣接するトークンペアの出現回数をカウントする。"""
        counts = Counter()
        for ids in token_ids_list:
            for a, b in zip(ids, ids[1:]):
                counts[(a, b)] += 1
        return counts

    def _merge(self, ids, pair, new_id):
        """ids 内のペアを new_id ですべて結合する。"""
        merged = []
        i = 0
        while i < len(ids):
            if i < len(ids) - 1 and (ids[i], ids[i + 1]) == pair:
                merged.append(new_id)
                i += 2
            else:
                merged.append(ids[i])
                i += 1
        return merged

    def train(self, text, vocab_size=500):
        """テキストから BPE 語彙辞書を学習する。"""
        # ステップ 1: 初期語彙は UTF-8 バイト 256 個
        for i in range(256):
            self.vocab[i] = bytes([i])

        # テキストをバイト単位の整数リストに変換
        ids = list(text.encode("utf-8"))
        token_ids_list = [ids]  # ドキュメントが一つのみのため、リスト内に一つだけ保持

        num_merges = vocab_size - 256
        for step in range(num_merges):
            pair_counts = self._get_pair_counts(token_ids_list)
            if not pair_counts:
                break
            best_pair = max(pair_counts, key=pair_counts.get)
            new_id = 256 + step
            self.vocab[new_id] = self.vocab[best_pair[0]] + self.vocab[best_pair[1]]
            self.merges[best_pair] = new_id
            token_ids_list = [self._merge(ids, best_pair, new_id) for ids in token_ids_list]

        return self

    def encode(self, text):
        """テキストをトークン ID リストに変換する。"""
        ids = list(text.encode("utf-8"))
        while len(ids) >= 2:
            pair_counts = self._get_pair_counts([ids])
            # 学習時に作成されたマージのうち、出現したペアの中で最も早く作成されたものを優先的に適用
            candidates = [p for p in pair_counts if p in self.merges]
            if not candidates:
                break
            best_pair = min(candidates, key=lambda p: self.merges[p])
            ids = self._merge(ids, best_pair, self.merges[best_pair])
        return ids

    def decode(self, ids):
        """トークン ID リストを再びテキストに復元する。"""
        byte_seq = b"".join(self.vocab[i] for i in ids)
        return byte_seq.decode("utf-8", errors="replace")


それでは、このトークナイザーを学習させ、実際にエンコーディングとデコーディングを行ってみましょう。


In [ ]:
# ── Colab セル ──
from mini_gpt.tokenizer import BPETokenizer

sample_text = """
人工知能は、人間の学習能力、推論能力、知覚能力を人工的に実装しようとするコンピュータ科学の細分化された分野である。
大規模言語モデルは、膨大なテキストデータを学習し、次の単語を予測する方式で言語を習得する。
Transformer 構造は、アテンション・メカニズムを核心として使用する。
""" * 20  # マージパターンが顕著に現れるよう繰り返し実行

tokenizer = BPETokenizer()
tokenizer.train(sample_text, vocab_size=500)

test = "Transformer wa attention o tsukau." # Transformer はアテンションを使用する。
encoded = tokenizer.encode(test)
decoded = tokenizer.decode(encoded)

print("原文:", test)
print("トークン ID 数:", len(encoded))
print("トークン ID:", encoded)
print("復元されたテキスト:", decoded)
print("原文と一致するか?:", test == decoded)


バイト単位で開始するため、日本語、英語、絵文字、特殊文字など、どのようなテキストが入力されても処理できる点がこの方式の大きな利点です。繰り返し実行してみると、頻繁に登場する日本語の断片（例：「Gengo Moderu (言語モデル)」、「Transformer (トランスフォーマー)」）が一つのトークンとして結合されていることが `tokenizer.vocab` から確認できます。

## 実戦では tiktoken を使用します

私たちが作成したトークナイザーは、原理を理解するための教育用です。実際には、学習速度がはるかに速く、すでに大規模なデータで語彙辞書が構築されているライブラリを使用します。OpenAIが公開している `tiktoken` がその代表例です。


In [ ]:
# ── Colab セル ──
import tiktoken

enc = tiktoken.get_encoding("gpt2")
test = "Transformers use attention mechanisms."
ids = enc.encode(test)
print("トークン ID:", ids)
print("トークン数:", len(ids))
print("復元:", enc.decode(ids))

# トークン単位で分割された断片を確認
for tid in ids:
    print(tid, "->", repr(enc.decode([tid])))


この後の6〜7章でミニGPTを学習させる際は、作成した`BPETokenizer`をそのまま使用して原理を維持しますが、実際のサービスでは`tiktoken`のような検証済みのライブラリを使用するという点を覚えておいてください。

次の章では、このように取得したトークンIDを、モデルが意味を持って扱える**ベクトル**へと変換する埋め込み層（Embedding layer）を作成していきます。


# 埋め込み（Embedding）と位置エンコーディング（Positional Encoding）

## トークンIDだけでは不十分な理由

トークナイザーを通ると、テキストは `[1023, 45, 892, ...]` のような整数のリストになります。しかし、整数そのものには意味がありません。トークンIDの1023と1024が隣り合う数字だからといって、これら2つのトークンの意味が似ているわけでは全くないのです。

そのため、各トークンIDを**意味を持つ実数ベクトル**に変換するプロセスが必要です。これが**埋め込み（Embedding）**です。埋め込みベクトルは学習を通じて、「意味の似たトークンはベクトル空間上で近い位置に」配置されるよう調整されます。例えば、学習がうまく進むと「猫」と「犬」の埋め込みベクトルは、「猫」と「車」よりも互いに近い位置になります。

実装の観点では、埋め込み層は実質的に単純な**ルックアップテーブル（lookup table）**です。語彙のサイズが $V$、埋め込み次元が $d$ である場合、$(V, d)$ サイズの行列を作成し、トークンIDをその行列の行番号として使用して該当するベクトルを取り出します。

## 位置情報が別途必要な理由

Transformerのアテンション・メカニズムは、文内のすべてのトークンを同時に並列処理します。これは速度面では大きな利点ですが、副作用として**「どのトークンが何番目にあるか」という情報が自動的には与えられません。** RNNは順序に従って処理するため位置情報が自然に反映されますが、Transformerは順序情報がないと「私はご飯を食べた」と「ご飯を私は食べた」を区別できません。

そこで、各位置（1番目、2番目、3番目...）ごとに固有のベクトルを加算する**位置エンコーディング（Positional Encoding）**を使用します。手法はいくつかあります。

- **サイン/コサイン関数ベース**（オリジナルのTransformer論文）: 位置ごとに異なる周波数のサイン・コサイン値を計算して使用。学習パラメータを持たない。
- **学習型位置埋め込み**（GPT-2など）: 位置インデックスもトークンと同様に、一つの埋め込みテーブルとして学習。

本書では、GPT-2と同じ方式である「学習型位置埋め込み」を使用します。実装がよりシンプルで理解しやすいためです。

## 実装する


In [ ]:
# ── Colab セル ──
%%writefile mini_gpt/embedding.py
"""トークン埋め込み + 位置埋め込み"""
import torch
import torch.nn as nn


class GPTEmbedding(nn.Module):
    def __init__(self, vocab_size, embed_dim, max_seq_len, dropout=0.1):
        super().__init__()
        # トークン ID -> ベクトル (ルックアップテーブル), サイズ: (vocab_size, embed_dim)
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        # 位置インデックス -> ベクトル, サイズ: (max_seq_len, embed_dim)
        self.position_embedding = nn.Embedding(max_seq_len, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, token_ids):
        # token_ids: (batch_size, seq_len)
        batch_size, seq_len = token_ids.shape
        positions = torch.arange(seq_len, device=token_ids.device).unsqueeze(0)
        # positions: (1, seq_len) -> ブロードキャストによりバッチ全体に適用

        token_vecs = self.token_embedding(token_ids)      # (batch, seq_len, embed_dim)
        position_vecs = self.position_embedding(positions)  # (1, seq_len, embed_dim)

        # 2つのベクトルを足し合わせることで「このトークンであり、かつこの位置である」という情報を共に保持する
        embeddings = token_vecs + position_vecs
        return self.dropout(embeddings)


## 直接動作を確認する

実際に動かして確認してみましょう。


In [ ]:
# ── Colab セル ──
from mini_gpt.embedding import GPTEmbedding
import torch

vocab_size = 500      # 第2章で作ったトークナイザーの語彙サイズ
embed_dim = 64         # 各トークンを表すベクトルの次元数
max_seq_len = 32        # モデルが一度に扱える最大トークン数

embedding_layer = GPTEmbedding(vocab_size, embed_dim, max_seq_len)

# ダミーのトークン ID バッチ: (batch_size=2, seq_len=5)
fake_token_ids = torch.tensor([
    [10, 25, 3, 88, 42],
    [7,  7,  7, 7,  7],
])

output = embedding_layer(fake_token_ids)
print("入力 shape:", fake_token_ids.shape)
print("出力 shape:", output.shape)   # (2, 5, 64) である必要がある
print("最初の文章、最初のトークンの埋め込みベクトルの一部:", output[0, 0, :8])


`fake_token_ids` の2文目は、同じトークン `7` が5回繰り返されています。もし位置エンベディング（position embedding）がなければ、5つの位置の出力ベクトルは完全に同一になっていたはずです。以下のコードを使って、実際に確認してみましょう。


In [ ]:
# ── Colab セル ──
# 同じトークンでも位置が異なれば埋め込み結果が変わるかを確認
second_sentence_output = output[1]  # (5, 64)
for i in range(5):
    print(f"位置 {i} のベクトルの一部:", second_sentence_output[i, :4].tolist())


5つのベクトルがすべて異なる値で出力されます。これこそが、位置エンコーディング（Positional Encoding）が「順序情報」を埋め込み（Embedding）に注入する仕組みです。

## 埋め込み次元（embed_dim）を決定する基準

`embed_dim` は、1つのトークンをいくつの実数で表現するかを決定するハイパーパラメータです。値が大きいほどより豊かな意味を保持できますが、計算量とメモリ使用量も増加します。参考までに、実際のモデルの概算規模は以下の通りです。

| モデル | 埋め込み次元 | レイヤー数 |
|---|---|---|
| GPT-2 Small | 768 | 12 |
| GPT-2 XL | 1600 | 48 |
| 本書のミニGPT | 64~128 | 4~6 |

本書では、Colabの無料GPUでも数分で学習が終了するように、非常に小さな規模で設計しています。原理は実際の大型モデルと完全に同一であり、次元数とレイヤー数を増やすだけでそのまま拡張可能です。

次章では、これらの埋め込みベクトルが互いに情報をやり取りするための核心的なメカニズムである**アテンション（Attention）**を実装します。


# アテンション・メカニズム

## コアとなるアイデア：文脈によって意味が変化する

「橋を渡る」「箸で食べる」「話の端をつかむ」のように、日本語では同じ読みの「はし」でも、文脈によって「橋」「箸」「端」というまったく異なる意味になります。埋め込み（Embedding）だけでは、同じ表記や同じ読みに対して常に似たベクトルを割り当ててしまい、この違いを十分に表現できません。

**アテンション（Attention）**は、各単語が文中の他の単語を「どれくらい参照するか」の重みを計算し、文脈に合わせてベクトルを再構成するメカニズムです。「はし」の周囲に「渡る」があれば「橋」に、「食べる」があれば「箸」に、「話」があれば「端」に近い意味として解釈できるよう、関連する語に集中（attend）することを学習します。

## Query, Key, Value の比喩

アテンションは、図書館で本を探すプロセスに例えることができます。

- **Query（クエリ）**: 「今、こういう情報が必要だ」という問いのベクトル。現在処理中の単語が生成します。
- **Key（キー）**: 各単語が「私はこのような情報を持っている」と提示するインデックス（索引）ベクトル。
- **Value（バリュー）**: 実際に渡される内容が含まれたベクトル。

Queryとすべての単語のKeyを比較（内積）して類似度スコアを計算し、このスコアを確率のように正規化（softmax）した後、その確率を用いてValueを重み付き加算します。つまり、「問い（Query）との関連度が高い単語の内容をより多く反映させて、新しいベクトルを作る」というのがアテンションのすべてです。

数式で表すと以下のようになります。

```
Attention(Q, K, V) = softmax( Q · K^T / sqrt(d_k) ) · V
```

- `Q · K^T`: QueryとKeyの内積により、類似度スコア行列を計算
- `sqrt(d_k)` で割る理由: 次次元が大きくなるほど内積の値が大きくなり、softmaxが一方に偏るのを防ぐため（スケーリング）
- `softmax`: スコアを合計が1となる確率分布に変換
- 最後に `V` を掛け、確率による加重平均を用いて新しいベクトルを計算

## Self-Attentionの実装


In [ ]:
# ── Colab セル ──
%%writefile mini_gpt/attention.py
"""Self-AttentionとMulti-Head Attentionの実装"""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class SelfAttention(nn.Module):
    """最も基本的なシングルヘッド Self-Attention"""

    def __init__(self, embed_dim, head_dim, max_seq_len, dropout=0.1):
        super().__init__()
        # Q, K, Vを作成する線形変換。入力と出力の次元が異なる場合がある(head_dim)
        self.query_proj = nn.Linear(embed_dim, head_dim, bias=False)
        self.key_proj = nn.Linear(embed_dim, head_dim, bias=False)
        self.value_proj = nn.Linear(embed_dim, head_dim, bias=False)
        self.dropout = nn.Dropout(dropout)

        # 未来のトークンを見ないようにするための causal mask (下三角行列)
        # GPTは「次の単語予測」が目的であるため、未来の情報を参照すると反則（チーティング）になります。
        causal_mask = torch.tril(torch.ones(max_seq_len, max_seq_len))
        self.register_buffer("causal_mask", causal_mask)

    def forward(self, x):
        # x: (batch, seq_len, embed_dim)
        batch_size, seq_len, _ = x.shape

        Q = self.query_proj(x)   # (batch, seq_len, head_dim)
        K = self.key_proj(x)     # (batch, seq_len, head_dim)
        V = self.value_proj(x)   # (batch, seq_len, head_dim)

        head_dim = Q.shape[-1]
        # 類似度スコア: (batch, seq_len, seq_len)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(head_dim)

        # 未来の位置は -inf でマスキング -> softmax 後に確率が 0 になる
        mask = self.causal_mask[:seq_len, :seq_len]
        scores = scores.masked_fill(mask == 0, float("-inf"))

        attn_weights = F.softmax(scores, dim=-1)  # 各行の合計が 1
        attn_weights = self.dropout(attn_weights)

        output = attn_weights @ V  # (batch, seq_len, head_dim)
        return output, attn_weights


class MultiHeadAttention(nn.Module):
    """複数のヘッドが異なる視点から並列でアテンションを計算"""

    def __init__(self, embed_dim, num_heads, max_seq_len, dropout=0.1):
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dimはnum_headsで割り切れる必要があります"
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.qkv_proj = nn.Linear(embed_dim, embed_dim * 3, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

        causal_mask = torch.tril(torch.ones(max_seq_len, max_seq_len))
        self.register_buffer("causal_mask", causal_mask)

    def forward(self, x):
        batch_size, seq_len, embed_dim = x.shape

        # 一度に Q, K, V をすべて計算 (効率化のため一つの線形層で処理)
        qkv = self.qkv_proj(x)  # (batch, seq_len, embed_dim*3)
        Q, K, V = qkv.chunk(3, dim=-1)

        # (batch, seq_len, embed_dim) -> (batch, num_heads, seq_len, head_dim)
        def split_heads(t):
            t = t.view(batch_size, seq_len, self.num_heads, self.head_dim)
            return t.transpose(1, 2)

        Q, K, V = split_heads(Q), split_heads(K), split_heads(V)

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.head_dim)
        mask = self.causal_mask[:seq_len, :seq_len]
        scores = scores.masked_fill(mask == 0, float("-inf"))

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        out = attn_weights @ V  # (batch, num_heads, seq_len, head_dim)

        # 再び (batch, seq_len, embed_dim) に結合
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, embed_dim)
        return self.out_proj(out)


## 動作確認とアテンション重みの可視化


In [ ]:
# ── Colab セル ──
from mini_gpt.attention import SelfAttention, MultiHeadAttention
import torch

embed_dim = 32
seq_len = 6
batch_size = 1

x = torch.randn(batch_size, seq_len, embed_dim)

attn = SelfAttention(embed_dim, head_dim=embed_dim, max_seq_len=16)
output, weights = attn(x)

print("入力 shape:", x.shape)
print("出力 shape:", output.shape)
print("アテンション重み shape:", weights.shape)  # (1, 6, 6)
print("\nアテンション重み行列 (行=現在のトークン, 列=参照するトークン):")
for row in weights[0]:
    print([round(v, 2) for v in row.tolist()])


出力された行列を見ると、上三角部分（未来の位置）がすべて0であることが確認できます。これがまさに `causal_mask` が未来の情報を遮断した結果です。各行の合計は正確に1になりますが、これは softmax が確率分布を作成するためです。


In [ ]:
# ── Colab セル ──
# Multi-Head Attentionも同様に確認
mha = MultiHeadAttention(embed_dim=32, num_heads=4, max_seq_len=16)
out = mha(x)
print("Multi-Head Attention 出力 shape:", out.shape)  # (1, 6, 32) - 入力と同じ shape を維持


## ヘッドを複数使用する理由

1つのアテンション・ヘッドは、単一の観点からのみ単語間の関係を計算します。例えば、あるヘッドは文法的な関係（主語と動詞）に集中し、別のヘッドは意味的な類似性に注目するといった使い分けが可能です。**Multi-Head Attention**は、埋め込み次元を複数の断片（head）に分割し、各断片が独立してアテンションを計算した後にそれらを再び結合します。これにより、モデルは複数の種類の関係を同時に並列的に捉えることができます。

次の章では、このアテンションを包み込み、実際に学習が安定するように構成する**トランスフォーマー・ブロック**(LayerNorm, FeedForward, Residual Connection)を実装します。


# Transformer ブロックの組み立て

前の章で作った Multi-Head Attention は、それ単体では学習がうまく進みません。実際の Transformer では、アテンションの周囲に 3 つの仕組みを追加して使用します：**残差接続 (Residual Connection)**、**レイヤー正規化 (Layer Normalization)**、そして **フィードフォワード・ネットワーク (FeedForward Network)** です。これら 3 つをアテンションと組み合わせた一つの単位を **Transformer ブロック** と呼び、このブロックを何層にも積み重ねることでモデルを構築します。

## 残差接続 (Residual Connection): 情報の消失を防ぐ

レイヤーを深く積み重ねるほど、元の情報は幾度もの変換を経て薄れてしまい、学習中に勾配（グラディエント）が消失するという問題が発生します。残差接続は、シンプルなトリックによってこの問題を解決します。```
output = attention(x) + x
```

つまり、変換した結果をそのまま使うのではなく、**入力 $x$ をもう一度足し合わせます。** こうすることで、たとえ最悪のケース（アテンションが全く役に立たない場合）であっても、少なくとも入力情報はそのまま次の層へと伝達されます。これは、いわば「ショートカット」をもう一つ作るようなものです。ResNetという画像モデルで初めて提案され、Transformerでもそのまま採用されています。

## レイヤー正規化 (Layer Normalization): 学習を安定させる

ニューラルネットワークが深くなると、各層を通過するたびに値の大きさ（スケール）が激しく変動し、学習が不安定になります。LayerNormは、各トークンのベクトルを平均 0、分散 1 となるように正規化した後、学習可能なスケールと移動値を掛け合わせ、加算します。```
LayerNorm(x) = γ · (x - mean) / sqrt(variance + ε) + β
```

ここで $\gamma$ (ガンマ) と $\beta$ (ベータ) は学習されるパラメータであり、正規化の後にモデルが必要に応じてスケールを再調整できるようにします。

## フィードフォワードネットワーク：トークンごとの非線形変換

アテンションが「トークン間の関係」を計算する役割を担うのに対し、フィードフォワードネットワークは各トークンのベクトルを独立して、より深く加工する役割を担います。その構造は単純です。次元を大きく拡張し（通常は4倍）、再び元の次元へと縮小させる2つの線形層の間に、非線形活性化関数 (GELU) を挿入します。```
FFN(x) = Linear2( GELU( Linear1(x) ) )
```

次元を拡大してから縮小させる理由は、中間層のより広い空間において多様な特徴を組み合わせる余裕を与えるためです。

## Pre-LN構造

LayerNormをどこに配置するかによって、Post-LNとPre-LNの2つの方式があります。オリジナルのTransformer論文ではPost-LN（Attention/FFNの後に正規化）が採用されましたが、GPT-2以降のほとんどのモデルは**Pre-LN**（Attention/FFNの前に正規化）方式を使用しています。これは、Pre-LNの方が学習初期において非常に安定しているためです。本書でもPre-LNを採用します。```
x = x + Attention(LayerNorm(x))
x = x + FFN(LayerNorm(x))
```

## 実装する


In [ ]:
# ── Colab セル ──
%%writefile mini_gpt/transformer_block.py
"""Transformer ブロック: Attention + FeedForward + LayerNorm + Residual"""
import torch.nn as nn
from mini_gpt.attention import MultiHeadAttention


class FeedForward(nn.Module):
    def __init__(self, embed_dim, hidden_mult=4, dropout=0.1):
        super().__init__()
        hidden_dim = embed_dim * hidden_mult
        self.net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, max_seq_len, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attention = MultiHeadAttention(embed_dim, num_heads, max_seq_len, dropout)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.feed_forward = FeedForward(embed_dim, hidden_mult=4, dropout=dropout)

    def forward(self, x):
        # Pre-LN 構造: 正規化 -> サブレイヤー -> 残差接続
        x = x + self.attention(self.ln1(x))
        x = x + self.feed_forward(self.ln2(x))
        return x


## 動作確認


In [ ]:
# ── Colab セル ──
from mini_gpt.transformer_block import TransformerBlock
import torch

embed_dim = 64
seq_len = 10
batch_size = 2

x = torch.randn(batch_size, seq_len, embed_dim)
block = TransformerBlock(embed_dim=embed_dim, num_heads=4, max_seq_len=32)

output = block(x)
print("入力 shape:", x.shape)
print("出力 shape:", output.shape)  # 入力と同じである必要がある - ブロックを多層に積み重ねられる理由
print("パラメータ数:", sum(p.numel() for p in block.parameters()))


ブロックの入力と出力の shape が完全に一致しているという点が重要です。そうすることで、このブロックをレゴブロックのように、好きな数だけ積み重ねることができるようになります。


In [ ]:
# ── Colab セル ──
# 残差接続が実際に勾配を安定させるか、簡単に確認
import torch.nn as nn

# ブロックを12層深く積み重ねても勾配消失が起きないかを確認
blocks = nn.ModuleList([
    TransformerBlock(embed_dim=64, num_heads=4, max_seq_len=32) for _ in range(12)
])

x = torch.randn(1, 10, 64, requires_grad=True)
out = x
for b in blocks:
    out = b(out)

loss = out.sum()
loss.backward()

print("入力 x の勾配の絶対値平均:", x.grad.abs().mean().item())
print("-> 0 に近くないということは、12層を通っても勾配が適切に伝達されていることを意味します。")


次章では、このトランスフォーマー・ブロックをN個積み重ね、エンベディング層と出力層を接続して、完全なミニGPTモデルを組み立てます。


# ミニ GPT モデルの組み立て

これまで作成したコンポーネントをまとめると、以下のようになります。

- `GPTEmbedding`: トークン ID → ベクトル (トークン埋め込み + 位置埋め込み)
- `TransformerBlock`: アテンション + フィードフォワード (これを $N$ 個積み重ねる)
- 最後に必要なもの: トランスフォーマー・ブロックの出力を、再び**「語彙サイズ分の確率分布」**へと変換する出力層

この章では、これらのコンポーネントを組み立てて、完全な GPT モデルクラスを作成します。

## 出力層 (LM Head)

トランスフォーマー・ブロックを通過した後、各トークン位置のベクトルは依然として `embed_dim` のサイズです。このベクトルを「次のトークンが語彙集の各単語である確率」に変換するには、`embed_dim` → `vocab_size` へと変換する線形層が必要です。これを **LM Head (Language Modeling Head)** と呼びます。```
logits = Linear(embed_dim -> vocab_size)(Transformer出力)
```

ロジットに softmax を適用すると、語彙全体に対する確率分布が得られます。実際の学習では、softmax を明示的に適用せず、`CrossEntropyLoss` が内部で softmax と対数（log）をまとめて計算するように任せます（この方が数値的に安定するためです）。

## 重み共有 (Weight Tying)

GPT-2 を含む多くのモデルでは、トークン埋め込み行列（Token Embedding Matrix）と LM Head の重みを**共有**しています。これらはどちらも「トークン ↔ ベクトル」の変換を担当する役割が似ているためです（一方は ID → ベクトル、もう一方は ベクトル → 各 ID のスコア）。これにより、パラメータ数を大幅に削減しつつ、性能をほぼ維持、あるいはむしろ向上させることが多くあります。この本のミニ GPT もこの手法を採用しています。

## 実装する


In [ ]:
# ── Colab セル ──
%%writefile mini_gpt/model.py
"""Mini GPT: Embedding + Transformer Blocks N個 + 出力層"""
import torch
import torch.nn as nn
from mini_gpt.embedding import GPTEmbedding
from mini_gpt.transformer_block import TransformerBlock


class MiniGPT(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, num_heads=4,
                 num_layers=4, max_seq_len=128, dropout=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len

        self.embedding = GPTEmbedding(vocab_size, embed_dim, max_seq_len, dropout)

        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, max_seq_len, dropout)
            for _ in range(num_layers)
        ])

        self.final_ln = nn.LayerNorm(embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size, bias=False)

        # 重み共有: トークン埋め込みと出力層が同じ行列を使用
        self.lm_head.weight = self.embedding.token_embedding.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        # GPT-2に類似した初期化: 小さな標準偏差の正規分布
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, token_ids, targets=None):
        # token_ids: (batch, seq_len)
        x = self.embedding(token_ids)          # (batch, seq_len, embed_dim)

        for block in self.blocks:
            x = block(x)

        x = self.final_ln(x)
        logits = self.lm_head(x)               # (batch, seq_len, vocab_size)

        loss = None
        if targets is not None:
            # logits: (batch, seq_len, vocab_size) -> (batch*seq_len, vocab_size)
            # targets: (batch, seq_len) -> (batch*seq_len,)
            loss = nn.functional.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
            )

        return logits, loss

    def num_parameters(self):
        return sum(p.numel() for p in self.parameters())


## モデルの生成とパラメータ数の確認

モデルを定義した後、そのモデルが持つパラメータ（重み）の総数を把握することは、メモリ使用量や計算コストを予測する上で非常に重要です。以下に、PyTorch を使用してモデルのパラメータ数を確認する方法を示します。

### 1. パラメータ総数の計算

モデル内のすべてのパラメータの要素数を合計することで、総パラメータ数を算出できます。


In [ ]:
def count_parameters(model):
    # すべてのパラメータの要素数（numel）を合計する
    return sum(p.numel() for p in model.parameters())

total_params = count_parameters(model)
print(f"Total Parameters: {total_params:,}")


### 2. 学習可能な（Trainable）パラメータのみのカウント

転移学習やファインチューニングを行う場合、一部の層をフリーズ（固定）させることがあります。この場合、実際に更新される「学習可能なパラメータ」の数を把握する必要があります。


In [ ]:
def count_trainable_parameters(model):
    # requires_grad=True のパラメータのみをカウントする
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

trainable_params = count_trainable_parameters(model)
print(f"Trainable Parameters: {trainable_params:,}")


### 3. 層ごとのパラメータ数を確認する（詳細）

より詳細な分析が必要な場合は、各レイヤーの名前とそのレイヤーが持つパラメータ数を一覧表示させると便利です。

| 方法 | 特徴 |
| :--- | :--- |
| `model.parameters()` | モデル全体の全パラメータをイテレートする |
| `model.named_parameters()` | パラメータ名（`layer1.weight` など）と共に取得できる |


In [ ]:
# 各レイヤーのパラメータ数を表示
for name, param in model.named_parameters():
    print(f"{name}: {param.numel():,}")


### まとめ

*   **総パラメータ数**: モデル全体のサイズ（メモリ容量）を決定します。
*   **学習可能パラメータ数**: 学習時の計算量と、更新される重みの範囲を決定します。
*   **`numel()`**: 各テンソル内の要素数を取得する PyTorch の便利な関数です。


In [ ]:
# ── Colab セル ──
from mini_gpt.model import MiniGPT
import torch

model = MiniGPT(
    vocab_size=500,     # Tokenizerの語彙サイズ
    embed_dim=128,
    num_heads=4,
    num_layers=4,
    max_seq_len=64,
    dropout=0.1,
)

print(model)
print("\n総パラメータ数:", f"{model.num_parameters():,}")


## 順伝播 (Forward Pass) の確認


In [ ]:
# ── Colab セル ──
# ダミーバッチで順伝播が正常に動作するか確認
batch_size = 4
seq_len = 16

fake_input = torch.randint(0, 500, (batch_size, seq_len))
fake_target = torch.randint(0, 500, (batch_size, seq_len))

logits, loss = model(fake_input, targets=fake_target)

print("入力 shape:", fake_input.shape)
print("logits shape:", logits.shape)        # (4, 16, 500)
print("損失値(loss):", loss.item())
print("\n損失値の解釈: 学習前のランダム初期化状態であるため、-log(1/500) =",
      round(torch.log(torch.tensor(500.0)).item(), 3), "付近になるのが正常です。")


損失値が `log(vocab_size)` 付近の値を示している場合、それはモデルが正常に初期化されたというシグナルです。これは、学習前のモデルがすべてのトークンに対して一様な確率（1/500）を予測していることを意味します。もしこの値がこれよりも大幅に大きかったり `nan` が出力されたりする場合は、初期化または実装にバグがあるというシグナルであるため、ここで確認して進むことが重要です。

## モデル規模を変更して実験する


In [ ]:
# ── Colab セル ──
# ハイパーパラメータを変更しながら、パラメータ数がどのように変化するか比較
configs = [
    {"embed_dim": 64,  "num_heads": 2, "num_layers": 2},
    {"embed_dim": 128, "num_heads": 4, "num_layers": 4},
    {"embed_dim": 256, "num_heads": 8, "num_layers": 6},
]

for cfg in configs:
    m = MiniGPT(vocab_size=500, max_seq_len=64, **cfg)
    print(cfg, "-> パラメータ数:", f"{m.num_parameters():,}")


`embed_dim` や `num_layers` を大きくすると、パラメータ数がどのように増加するかを直接観察してみましょう。この関係性を理解することで、実際の GPT-3 (1750億) や GPT-4 といったモデルが、なぜこれほど膨大なパラメータを持つのか（層を非常に深く、次元を非常に広く積み重ねているため）という感覚を掴むことができます。

次の章では、このモデルを実際のテキストデータで学習させるための学習ループを実装します。


# 学習ループの実装

それでは、ミニGPTを実際のテキストで学習させてみましょう。学習ループの核心となるアイデアはシンプルです。**「あるテキストの一片をモデルに入力し、各位置で『次に来るトークン』を予測させ、間違えた分だけ重みを少しずつ修正する」**というプロセスを、数百から数千回繰り返します。

## 学習データの準備：入力と正解を1つずつずらす

言語モデルの学習において、正解ラベル（教師データ）は人間が別途作成する必要はありません。元のテキストそのものが正解となります。例えば、トークンシーケンスが `[私, は, ご飯, を, 食べ, た]` である場合：

- 入力: `[私, は, ご飯, を, 食べ]`
- 正解: `[は, ご飯, を, 食べ, た]` （入力を1つ左にずらしたもの）

つまり、入力の $i$ 番目の位置において、モデルは正解の $i$ 番目のトークン（＝入力の $i+1$ 番目のトークン）を予測しなければなりません。このように、ラベルを別途作成することなく元のテキストのみから大量の学習データを作成できることが、言語モデルの事前学習における大きな利点です（これは自己教師あり学習、self-supervised learning と呼ばれます）。

## データセットクラスの実装


In [ ]:
# ── Colab セル ──
%%writefile mini_gpt/dataset.py
"""テキスト -> (入力, 正解) 学習バッチを作成するデータセット"""
import torch
from torch.utils.data import Dataset


class TextDataset(Dataset):
    def __init__(self, token_ids, seq_len):
        """token_ids: 全テキストをトークナイズした長い整数のリスト"""
        self.token_ids = torch.tensor(token_ids, dtype=torch.long)
        self.seq_len = seq_len

    def __len__(self):
        # seq_len+1 サイズのチャンクを重なりなく抽出できる開始位置の数
        return len(self.token_ids) - self.seq_len

    def __getitem__(self, idx):
        chunk = self.token_ids[idx: idx + self.seq_len + 1]
        x = chunk[:-1]   # 入力: 前方の seq_len 個
        y = chunk[1:]    # 正解: 1つずらした seq_len 個
        return x, y


## 学習データの準備 (Colabで実行)

今回は、Wikipediaの韓国語ドキュメントの一部を例題データとして使用します。演習用のため短い公開テキストを使用しますが、皆さんのプロジェクトでは任意のテキストファイルに変更しても構いません。


In [ ]:
# ── Colab セル ──
from datasets import load_dataset

# 実習用に WikiText の非常に小さな英語サブセットを使用 (ダウンロードが高速)
raw_dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:2%]")
text_corpus = "\n".join([t for t in raw_dataset["text"] if len(t.strip()) > 0])
print("総文字数:", len(text_corpus))
print("サンプル:\n", text_corpus[:500])


In [ ]:
# ── Colab セル ──
# 第2章で作った BPE トークナイザーをこのコーパスで学習
from mini_gpt.tokenizer import BPETokenizer

tokenizer = BPETokenizer()
tokenizer.train(text_corpus, vocab_size=2000)

all_token_ids = tokenizer.encode(text_corpus)
print("総トークン数:", len(all_token_ids))
print("語彙辞書サイズ:", len(tokenizer.vocab))


In [ ]:
# ── Colab セル ──
from mini_gpt.dataset import TextDataset
from torch.utils.data import DataLoader

seq_len = 64
dataset = TextDataset(all_token_ids, seq_len=seq_len)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

x_batch, y_batch = next(iter(dataloader))
print("入力バッチの shape:", x_batch.shape)   # (32, 64)
print("正解バッチの shape:", y_batch.shape)   # (32, 64)


## 学習ループ

学習の各ステップ（step）は、以下の5つの段階で構成されます。

1. **順伝播 (forward)**: モデルに入力を入れ、予測値（ロジット）と損失（loss）を計算
2. **逆伝播 (backward)**: 損失に基づき、各パラメータが損失にどれだけ寄与したか（勾配/gradient）を計算
3. **勾配の初期化**: 前のステップの勾配が累積されないよう、0で初期化
4. **オプティマイザのステップ (optimizer step)**: 計算された勾配の方向にパラメータを少しずつ移動
5. 次のバッチへ繰り返し


In [ ]:
# ── Colab セル ──
%%writefile mini_gpt/train.py
"""学習ループ"""
import torch


def train_model(model, dataloader, num_epochs, learning_rate, device):
    model.to(device)
    model.train()

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    # 学習が進むにつれて学習率を徐々に下げ、より安定した収束を実現する
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs * len(dataloader)
    )

    history = []
    step = 0
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        for x_batch, y_batch in dataloader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)

            logits, loss = model(x_batch, targets=y_batch)

            optimizer.zero_grad()
            loss.backward()
            # 勾配爆発の防止: 勾配の大きさを一定の範囲に制限
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            epoch_loss += loss.item()
            step += 1
            history.append(loss.item())

            if step % 50 == 0:
                print(f"epoch {epoch+1} | step {step} | loss {loss.item():.4f}")

        avg_loss = epoch_loss / len(dataloader)
        print(f"=== epoch {epoch+1} 平均 loss: {avg_loss:.4f} ===")

    return history


## 実際に学習させてみる

実際にモデルを学習させるプロセスを開始しましょう。ここでは、データセットの準備から学習の実行まで、一連の流れを説明します。


In [ ]:
# ── Colab セル ──
from mini_gpt.model import MiniGPT
from mini_gpt.train import train_model
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model = MiniGPT(
    vocab_size=len(tokenizer.vocab),
    embed_dim=128,
    num_heads=4,
    num_layers=4,
    max_seq_len=seq_len,
    dropout=0.1,
)
print("パラメータ数:", f"{model.num_parameters():,}")

history = train_model(
    model,
    dataloader,
    num_epochs=3,
    learning_rate=3e-4,
    device=device,
)


## 学習曲線の確認

学習曲線（Learning Curve）を確認することで、モデルが過学習（Overfitting）しているか、あるいは未学習（Underfitting）の状態にあるかを判断できます。

学習曲線は、訓練データに対する誤差と検証データに対する誤差を、エポック数（またはトレーニング回数）に対してプロットしたものです。

### 学習曲線の見方

*   **過学習 (Overfitting):** 訓練データの誤差が下がり続けている一方で、検証データの誤差が増加し始める状態です。これは、モデルが訓練データのノイズまで学習してしまい、未知のデータに対する汎化性能が低下していることを示します。
*   **未学習 (Underfitting):** 訓練データと検証データの両方の誤差が高いレベルで停滞している状態です。これは、モデルの表現力が不足しており、データのパターンを十分に捉えられていないことを示します。
*   **理想的な状態:** 訓練データと検証データの誤差が共に低くなり、かつその差（ギャップ）が小さい状態です。

以下のコードでは、学習曲線を描画するプロセスを示します。


In [ ]:
# ── Colab セル ──
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(history)
plt.xlabel("学習ステップ")
plt.ylabel("Loss")
plt.title("学習損失の推移")
plt.grid(True, alpha=0.3)
plt.show()


Lossが継続的に右肩下がりに減少している場合、モデルがデータのパターン（頻出する単語の組み合わせ、文法構造）を学習していることを意味します。初期のlossは `log(vocab_size)` 付近から始まり、学習が進むにつれて徐々に低下していくのが正常な挙動です。

## チェックポイントの保存


In [ ]:
# ── Colab セル ──
import pickle

torch.save(model.state_dict(), "mini_gpt_checkpoint.pt")
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("モデルとトークナイザーを保存しました。")


Colabセッションは一定時間が経過すると初期化されるため、重要なチェックポイントは `from google.colab import drive; drive.mount('/content/drive')` を使用してGoogle ドライブをマウントし、そのパスに保存しておくことを推奨します。

次章では、学習済みモデルを用いて実際にテキストを生成する方法と、生成結果の品質を左右するサンプリング戦略について扱います。


# テキスト生成とサンプリング戦略

## 生成は「次トークンの予測」の繰り返し

学習済みモデルは、与えられたトークンシーケンスに対して「次のトークンが何になるか」の確率分布（ロジット）を出力します。テキスト生成とは、このプロセスを繰り返すことです。

1. 現在までのトークンシーケンスをモデルに入力する。
2. 最後の位置のロジットを確率分布に変換する。
3. その分布から次のトークンを1つ選択する。
4. 選択したトークンをシーケンスの末尾に結合する。
5. 希望の長さに達するまで、1〜4を繰り返す。

ここで、3番の「分布からどのように次のトークンを選択するか」が、生成される文章の品質と創造性を大きく左右します。同じモデルであっても、サンプリング戦略によって全く異なる印象の文章が出力されます。

## サンプリング戦略

### Greedy Decoding（貪欲法）
毎回、確率が最も高いトークンのみを選択します。最も「安全な」文章が得られますが、繰り返しが多く退屈な文章になりやすい傾向があります。

### Temperature（温度）
softmaxを適用する前に、ロジットを温度の値で割ります。```
確率 = softmax(ロジット / temperature)
```

- `temperature < 1`: 確率分布がより尖る → 自信に満ちた（ただし単調な）出力
- `temperature > 1`: 確率分布がより平坦になる → 多様性は増すが、支離滅裂になるリスクが増加
- `temperature = 1`: モデルが学習した元の分布をそのまま使用

### Top-k サンプリング
確率が高い上位 $k$ 個のトークンのみを候補として残し、それ以外は確率を 0 にした上でその中からサンプリングします。文脈に合わない（意味不明な）トークンが低い確率であっても選ばれてしまうのを防ぎます。

### Top-p (Nucleus) サンプリング
累積確率が $p$（例: 0.9）を超えるまで、確率の高い順にトークンを候補に追加し、その中からサンプリングします。Top-k は候補の数を固定しますが、Top-p は分布が尖っているときは候補を少なく、平坦なときは候補を多く残すというように、状況に合わせて柔軟に調整されます。実際のサービスで最も広く使われている手法です。

## 実装する


In [ ]:
# ── Colab セル ──
%%writefile mini_gpt/generate.py
"""テキスト生成: greedy, temperature, top-k, top-p サンプリング"""
import torch
import torch.nn.functional as F


@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=50,
             temperature=1.0, top_k=None, top_p=None, device="cpu"):
    model.eval()
    model.to(device)

    token_ids = tokenizer.encode(prompt)
    token_ids = torch.tensor(token_ids, dtype=torch.long, device=device).unsqueeze(0)

    for _ in range(max_new_tokens):
        # モデルが学習時に見た最大長を超えないよう、直近のトークンのみを使用
        context = token_ids[:, -model.max_seq_len:]

        logits, _ = model(context)
        last_logits = logits[:, -1, :]  # 最後の位置のロジット: (batch, vocab_size)

        # temperature の適用
        last_logits = last_logits / max(temperature, 1e-5)

        # top-k フィルタリング: 上位 k 個以外は確率を -inf に
        if top_k is not None:
            topk_values, _ = torch.topk(last_logits, top_k)
            threshold = topk_values[:, -1].unsqueeze(-1)
            last_logits = torch.where(
                last_logits < threshold,
                torch.full_like(last_logits, float("-inf")),
                last_logits,
            )

        # top-p (nucleus) フィルタリング
        if top_p is not None:
            sorted_logits, sorted_idx = torch.sort(last_logits, descending=True)
            sorted_probs = F.softmax(sorted_logits, dim=-1)
            cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

            # 累積確率が top_p を超える地点から、除去対象としてマーク
            remove_mask = cumulative_probs > top_p
            # 最初のトークンは常に残し、候補が全くなくなるのを防ぐ
            remove_mask[:, 1:] = remove_mask[:, :-1].clone()
            remove_mask[:, 0] = False

            sorted_logits[remove_mask] = float("-inf")
            last_logits = torch.full_like(last_logits, float("-inf"))
            last_logits.scatter_(1, sorted_idx, sorted_logits)

        probs = F.softmax(last_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)  # 確率的サンプリング

        token_ids = torch.cat([token_ids, next_token], dim=1)

    generated_ids = token_ids[0].tolist()
    return tokenizer.decode(generated_ids)


## 実際に生成してみる


In [ ]:
# ── Colab セル ──
from mini_gpt.generate import generate

prompt = "The history of"

print("=== Greedy (temperatureを0に近づける) ===")
print(generate(model, tokenizer, prompt, max_new_tokens=40,
                temperature=0.01, device=device))

print("\n=== Temperature 1.0 ===")
print(generate(model, tokenizer, prompt, max_new_tokens=40,
                temperature=1.0, device=device))

print("\n=== Top-k サンプリング (k=20) ===")
print(generate(model, tokenizer, prompt, max_new_tokens=40,
                temperature=1.0, top_k=20, device=device))

print("\n=== Top-p サンプリング (p=0.9) ===")
print(generate(model, tokenizer, prompt, max_new_tokens=40,
                temperature=0.8, top_p=0.9, device=device))


この本のミニ GPT は、学習データとパラメータ数が非常に小さいため、文法的に完璧であったり事実に即した文章を期待することは困難です。しかし、学習が適切に進んでいれば、英単語の形態、頻出する組み合わせ、句読点の使用パターン程度は、それらしく模倣する様子を観察できます。この演習の目的は「実用レベルのモデルを作ること」ではなく、**同じ原理がどのように大規模に拡張され、ChatGPT のようなモデルへと進化するのかを体験すること**にあります。

## パラメータ別 体感比較


In [ ]:
# ── Colab セル ──
# 同じプロンプトでも、temperature によって結果がどれほど変化するかを比較
for temp in [0.3, 0.7, 1.2]:
    print(f"\n--- temperature={temp} ---")
    for _ in range(2):
        print(generate(model, tokenizer, prompt, max_new_tokens=30,
                        temperature=temp, top_p=0.9, device=device))


temperatureが低いほど2回の生成結果は互いに似通い、高いほど毎回より多様で予測不可能な結果が得られることが確認できます。

次の章では、私たちがゼロから構築したこの構造が、実際には数十億のパラメータを持つ事前学習済みモデルと原理的に同一であることを、Hugging Faceライブラリを用いて確認していきます。


# Hugging Face で本物の事前学習モデルを扱ってみる

これまで作成してきたミニ GPT は、パラメータ数が数十万〜数百万個レベルであり、学習データも非常に小規模なものでした。この章では、私たちが作ったものと**構造的に同一でありながら**、はるかに大規模で、より膨大なデータで学習された実際の公開モデルを Hugging Face の `transformers` ライブラリを使って読み込んでみます。自作した構造と比較することで、「規模が異なるだけで原理は同じである」ということを体感できるはずです。

## transformers ライブラリの基本操作


In [ ]:
# ── Colab セル ──
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "gpt2"  # 1億2千万パラメータの小型公開モデル

hf_tokenizer = AutoTokenizer.from_pretrained(model_name)
hf_model = AutoModelForCausalLM.from_pretrained(model_name)
hf_model.eval()

print("パラメータ数:", f"{sum(p.numel() for p in hf_model.parameters()):,}")
print(hf_model.config)


`hf_model.config` を出力すると、`n_embd`（埋め込み次元）、`n_layer`（レイヤー数）、`n_head`（ヘッド数）といった値が表示されます。これらは、私たちが `MiniGPT` を作成する際に使用した `embed_dim`、`num_layers`、`num_heads` と正確に対応しています。GPT-2 では、これらの値はそれぞれ 768、12、12 であり、私たちのミニ GPT よりもはるかに大きくなっています。

## 事前学習済みモデルによるテキスト生成


In [ ]:
# ── Colab セル ──
prompt = "The key idea behind the Transformer architecture is"
inputs = hf_tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    output_ids = hf_model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        top_k=50,
        pad_token_id=hf_tokenizer.eos_token_id,
    )

print(hf_tokenizer.decode(output_ids[0], skip_special_tokens=True))


`generate()` 関数の `temperature`、`top_p`、`top_k` 引数は、第8章で直接実装したものと名前まで同一です。私たちが作成した `generate()` 関数は、まさにこの関数の縮小版であったと言えます。

## モデルの内部構造を直接覗いてみる


In [ ]:
# ── Colab セル ──
# GPT-2 内部の Transformer ブロック一覧を確認
print(hf_model.transformer.h)  # h = "hidden layers", 我々の self.blocks と同じ役割

print("\nブロック数:", len(hf_model.transformer.h))
print("最初のブロックの構造:")
print(hf_model.transformer.h[0])


出力を見ると、`attn` (Attention)、`mlp` (我々の FeedForward)、`ln_1`, `ln_2` (我々の ln1, ln2) がそのまま表示されています。変数名や詳細な実装は多少異なりますが、第5章で作った `TransformerBlock` と全く同じ構成要素で構成されています。

## アテンションの重みを直接取り出して可視化


In [ ]:
# ── Colab セル ──
import matplotlib.pyplot as plt

inputs = hf_tokenizer("The cat sat on the mat because it was tired", return_tensors="pt")
with torch.no_grad():
    outputs = hf_model(**inputs, output_attentions=True)

# outputs.attentions: レイヤーごとの (batch, num_heads, seq_len, seq_len) テンソル・タプル
first_layer_attn = outputs.attentions[0][0]  # (num_heads, seq_len, seq_len)
tokens = hf_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for i, ax in enumerate(axes):
    ax.imshow(first_layer_attn[i].numpy(), cmap="viridis")
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=90, fontsize=8)
    ax.set_yticklabels(tokens, fontsize=8)
    ax.set_title(f"Head {i}")
plt.tight_layout()
plt.show()


各ヘッドが異なるパターンで単語同士を接続していることがわかります。例えば、あるヘッドは "it" が "cat" を指す代名詞の関係に強く反応したりします。これが、第4章で説明した「Multi-Head Attentionが複数の視点を同時に捉える」という言葉の実際の姿です。

## 本モデルとGPT-2の違いの要約

| 項目 | この本の MiniGPT | GPT-2 (small) |
|---|---|---|
| 埋め込み次元 (Embedding Dimension) | 64~128 | 768 |
| レイヤー数 | 4~6 | 12 |
| ヘッド数 | 4 | 12 |
| パラメータ数 | 数十万〜数百万 | 1.24億 |
| 学習トークン数 | 数万 | 約 100億 |
| 語彙辞書サイズ | 500~2000 | 50,257 |

表からわかるように、構造の「種類」は完全に同一であり、「規模」だけが異なります。これがLLMスケーリング（scaling）の核心です。同じ構造を維持したまま、データ、パラメータ、計算量を増大させると、性能が予測可能な形で向上するということが、近年のLLM研究において繰り返し確認されている経験則（スケーリング則、scaling law）です。

次の章では、このような事前学習モデルを一から学習し直すのではなく、少ないリソースで特定の目的に合わせて調整するファインチューニング、特に **LoRA** という軽量化手法について扱います。


# LoRAによる軽量なファインチューニング

## なぜフル・ファインチューニングは負担なのか

事前学習済みモデルを特定のタスク（例：特定の口調で回答する、特定のドメイン知識を反映させる）に適応させるには、追加の学習が必要です。最も直感的な方法は**モデルの全パラメータ**を再学習させることですが、これにはいくつかの現実的な問題があります。

- モデルが大きくなるほど、すべてのパラメータの勾配（gradient）とオプティマイザの状態（optimizer state）をGPUメモリに載せる必要があり、メモリ使用量が非常に大きくなります。
- ファインチューニングの結果ごとに、元のモデル全体のサイズと同じだけの保存容量が再び必要になります。
- データが少ない場合に全パラメータを変更すると、既存の学習済み知識を忘れてしまう**破滅的忘却（catastrophic forgetting）**が発生しやすくなります。

## LoRAの核心的なアイデア

**LoRA (Low-Rank Adaptation)** は、元のモデルの重みはそのまま凍結（freeze）した状態で、各レイヤーの横に非常に小さな2つの行列（A, B）を追加し、その小さな行列のみを学習させる手法です。

本来の重みの更新は、次のように表現できます。```
新しい重み = 元の重み W + ΔW
```

一般的なファインチューニングでは、`ΔW`は`W`と同じサイズの完全な行列です。LoRAは、`ΔW`を2つの非常に小さな行列の積によって近似します。```
ΔW ≈ B · A     (A: r×d サイズ, B: d×r サイズ, r は d よりはるかに小さい値, 例: r=8)
```

`r`(rank)が小さくなるほど、学習すべきパラメータ数は劇的に減少します。例えば、`d=768`のレイヤーを丸ごとファインチューニングする場合、`768×768 ≈ 59万`個のパラメータが必要ですが、`r=8`のLoRAでは`768×8 + 8×768 ≈ 1.2万`個となり、約2%のみを学習すれば済みます。驚くべきことに、多くのタスクにおいてこの手法はフルファインチューニングと同等の性能を発揮します。

これにより、以下のような利点が得られます。

- 学習すべきパラメータ数とGPUメモリの使用量が大幅に削減されます。
- 元の重みはそのまま保持されるため、LoRAアダプター（行列A, B）のみを保存すればよく、成果物の容量が数十MBレベルと非常に小さくなります。
- 複数のLoRAアダプターを作成しておき、状況に応じて切り替えて使用できます。

## Hugging Faceのpeftライブラリを用いた実践


In [ ]:
# ── Colab セル ──
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
import torch

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(model_name)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                       # LoRA rank: 小さいほどパラメータ数が少なくなる
    lora_alpha=16,             # スケーリング係数 (通常 r の2倍)
    lora_dropout=0.05,
    target_modules=["c_attn"],  # GPT-2 で Q, K, V を生成するアテンション層にのみ LoRA を適用
)

peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()


`print_trainable_parameters()` を実行すると、全パラメータ数に対する学習可能（trainable）なパラメータの割合が出力されます。通常、これは 1% 未満であることが確認できます。これが LoRA が「軽量な」ファインチューニングと呼ばれる理由です。

## シンプルなデータセットでファインチューニングする

ここでは例として、特定の話し方（簡潔で断定的な文体）で回答するように、少量の例題データを作成して学習を行います。


In [ ]:
# ── Colab セル ──
from torch.utils.data import Dataset, DataLoader

training_examples = [
    "Q: What is a transformer?\nA: A transformer is a neural network built entirely on attention.",
    "Q: What is attention?\nA: Attention lets a model weigh how much each token matters to another.",
    "Q: What is a tokenizer?\nA: A tokenizer converts raw text into integer tokens a model can process.",
    "Q: What is fine-tuning?\nA: Fine-tuning adapts a pretrained model to a new task using extra training.",
] * 30  # 小規模データを繰り返し学習させ、パターンを習得させる


class SimpleTextDataset(Dataset):
    def __init__(self, examples, tokenizer, max_len=64):
        self.encodings = tokenizer(
            examples, truncation=True, padding="max_length",
            max_length=max_len, return_tensors="pt",
        )

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        input_ids = self.encodings["input_ids"][idx]
        attention_mask = self.encodings["attention_mask"][idx]
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": input_ids.clone(),
        }


train_dataset = SimpleTextDataset(training_examples, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)


In [ ]:
# ── Colab セル ──
device = "cuda" if torch.cuda.is_available() else "cpu"
peft_model.to(device)
peft_model.train()

optimizer = torch.optim.AdamW(peft_model.parameters(), lr=1e-4)

for epoch in range(3):
    total_loss = 0.0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = peft_model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"epoch {epoch+1} 平均 loss: {total_loss/len(train_loader):.4f}")


## LoRA アダプターの保存と再利用

LoRA（Low-Rank Adaptation）を用いた学習では、モデル全体を保存するのではなく、追加された低ランク行列（アダプター）のみを保存することで、ストレージ容量を大幅に節約できます。以下に、学習したアダプターの保存方法と、それらをロードして推論に使用する方法を説明します。

### 1. アダプターの保存

`peft` ライブラリを使用すると、`save_pretrained()` メソッドを呼び出すだけで、アダプターの重み（`adapter_model.bin`）と設定ファイル（`adapter_config.json`）が指定したディレクトリに保存されます。


In [ ]:
# 学習済みのモデルとトークナイザーを保存する例
model.save_pretrained("./lora-adapter-output")
tokenizer.save_pretrained("./lora-adapter-output")


この方法で保存されたディレクトリには、以下のファイルが含まれます：
* `adapter_model.bin`: 学習された LoRA 重み。
* `adapter_config.json`: LoRA の設定（ランク $r$、$\alpha$ など）。

### 2. アダプターのロードと再利用

学習済みのアダプターを既存のベースモデルに適用するには、まずベースとなるモデルを読み込み、その後に `PeftModel.from_pretrained()` を使用してアダプターを重ね合わせます。


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# 1. ベースモデルのロード
base_model_id = "base-model-name" # 例: "meta-llama/Llama-2-7b-hf"
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# 2. 学習済みアダプターのロード
adapter_path = "./lora-adapter-output"
model = PeftModel.from_pretrained(model, adapter_path)

# 3. 推論の準備
tokenizer = AutoTokenizer.from_pretrained(adapter_path)
model.eval()

# テスト実行
inputs = tokenizer("Hello, my name is", return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=20)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


### 3. 複数のアダプターの切り替え (Multi-LoRA)

`peft` ライブラリは、一つのモデルに複数の LoRA アダプターをロードし、実行時にそれらを動的に切り替える機能をサポートしています。これは、一つのベースモデルを使用して様々なタスク（例：要約、翻訳、対話）を実行する際に非常に有用です。


In [ ]:
# 新しいアダプターの追加 (Base model の上に新しい LoRA レイヤーを追加)
model.load_adapter("./lora-adapter-task-A", adapter_name="task_A")
model.load_adapter("./lora-adapter-task-B", adapter_name="task_B")

# 'task_A' に切り替えて推論
model.set_adapter("task_A")
# ... inference with task_A ...

# 'task_B' に切り替えて推論 (モデルを再ロードする必要なし)
model.set_adapter("task_B")
# ... inference with task_B ...


### 要約テーブル

| タスク | 使用されるメソッド/クラス | 主な特徴 |
| :--- | :--- | :--- |
| **保存** | `model.save_pretrained()` | アダプターの重みと設定値のみを保存し、容量を最適化 |
| **ロード** | `PeftModel.from_pretrained()` | ベースモデルの上に学習済みの LoRA を結合 |
| **切り替え** | `model.set_adapter()` | ロードされた複数のアダプター間を高速に切り替え |


In [ ]:
# ── Colab セル ──
peft_model.save_pretrained("gpt2-lora-adapter")
print("アダプターの保存が完了しました。フォルダ容量を確認してください (元のモデルよりはるかに小さいです)。")

import os
total_size = sum(
    os.path.getsize(os.path.join("gpt2-lora-adapter", f))
    for f in os.listdir("gpt2-lora-adapter")
)
print(f"アダプターフォルダサイズ: {total_size / 1024 / 1024:.2f} MB")


In [ ]:
# ── Colab セル ──
# 保存したアダプターを後で再度読み込み、元のモデルに適用する方法
from peft import PeftModel

fresh_base_model = AutoModelForCausalLM.from_pretrained(model_name)
loaded_model = PeftModel.from_pretrained(fresh_base_model, "gpt2-lora-adapter")

loaded_model.eval()
prompt = "Q: What is a transformer?\nA:"
inputs = tokenizer(prompt, return_tensors="pt")
with torch.no_grad():
    output = loaded_model.generate(**inputs, max_new_tokens=30, do_sample=False)
print(tokenizer.decode(output[0], skip_special_tokens=True))


このように、LoRAは元のモデル（GPT-2、LLaMA系、Qwenなど、あらゆるモデル）をそのまま保持したまま、小さなアダプターのみを学習・保存・置換することができます。そのため、個人のGPUやColabの無料版のようにリソースが制限された環境でも、実践的なファインチューニングを体験できる実用的な手法です。

次の章では、モデルの重みを一切変更することなく、望む回答を引き出すための**プロンプトエンジニアリング**の手法について見ていきます。


# プロンプトエンジニアリングと推論時の制御

ファインチューニングはモデルの重みそのものを変更する方法でした。対して、**プロンプトエンジニアリング (Prompt Engineering)** は、重みには一切触れず、入力（プロンプト）を設計することによってモデルの出力を望む方向へと誘導する手法です。コストがほとんどかからず、かつ効果が高いため、実務において真っ先に試される方法です。

## Zero-shot vs Few-shot

- **Zero-shot**: 例示を与えず、指示のみを与える方式。`"次の文章を韓国語に翻訳してください: Hello"`
- **Few-shot**: 望ましい入出力形式の例をいくつかプロンプトに含める方式。モデルは例のパターンを見て、その形式に従います。


In [ ]:
# ── Colab セル ──
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

def generate_text(prompt, max_new_tokens=30, temperature=0.7):
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=True, temperature=temperature, top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)


zero_shot_prompt = "Translate to French: Good morning"
print("=== Zero-shot ===")
print(generate_text(zero_shot_prompt))

few_shot_prompt = (
    "English: Hello -> French: Bonjour\n"
    "English: Thank you -> French: Merci\n"
    "English: Good morning -> French:"
)
print("\n=== Few-shot ===")
print(generate_text(few_shot_prompt))


Few-shot プロンプトは、モデルに対して「今どのようなタスクを行っているのか」「回答をどのような形式で出力すべきか」を例示によって直接示すため、特に小規模なモデルにおいて Zero-shot よりも遥かに安定した結果をもたらします。これは、事前学習の段階でモデルが膨大なテキスト内のパターン反復をすでに学習しているからこそ可能な現象です。

## ロール（役割）の付与とシステムプロンプト

ChatGPT のような対話型モデルでは、対話の開始時に「あなたは有能なコーディングアシスタントです」といった **システムプロンプト** を挿入することで、モデルの役割や回答スタイルを指定します。その原理は Few-shot と似ています。モデルへの入力の冒頭にコンテキスト（文脈）を敷いておくことで、その後に続く生成がそのコンテキストに従う確率が高まるのです。


In [ ]:
# ── Colab セル ──
system_context = (
    "You are a concise technical writer. You answer in exactly one short sentence.\n\n"
)
question = "Q: What is gradient descent?\nA:"

print(generate_text(system_context + question, max_new_tokens=25))


## Chain-of-Thought (思考の連鎖) プロンプティング

複雑な推論を必要とする問題では、「すぐに答えを出せ」と指示するよりも「ステップごとに考えてみよ」と誘導した方が、より正確な回答が得られる傾向が観察されます。これを Chain-of-Thought (CoT) プロンプティングと呼びます。


In [ ]:
# ── Colab セル ──
direct_prompt = "Q: If a train travels 60 miles in 1.5 hours, what is its speed? A:"
cot_prompt = (
    "Q: If a train travels 60 miles in 1.5 hours, what is its speed?\n"
    "A: Let's think step by step. Speed equals distance divided by time. "
    "Distance is 60 miles, time is 1.5 hours. So speed ="
)

print("=== 直接質問 ===")
print(generate_text(direct_prompt, max_new_tokens=20))
print("\n=== ステップバイステップの思考誘導 ===")
print(generate_text(cot_prompt, max_new_tokens=20))


この本のミニGPTや小型のGPT-2では効果が限定的に見えるかもしれませんが、パラメータ規模が大きいモデルほどCoT（Chain-of-Thought）プロンプティングの効果が顕著に現れることが多くの研究で報告されています。これは、モデルの規模が大きくなるにつれて現れる「創発的能力（emergent ability）」の代表的な例として挙げられます。

## 推論時のパラメータ要約

これまで学んできた生成パラメータをまとめると以下の通りです。

| パラメータ | 役割 | 値を下げると | 値を上げると |
|---|---|---|---|
| `temperature` | 確率分布の鋭さを調整 | より決定的、反復的 | より多様、ランダム |
| `top_k` | 候補トークン数の制限 | 候補が少なく安全な選択 | より多様な候補を許容 |
| `top_p` | 累積確率による候補制限 | 候補が少なくなる | より広い候補を許容 |
| `max_new_tokens` | 生成する最大トークン数 | 短い応答 | 長い応答 (コスト/時間が増加) |

実務では、正解が明確なタスク（コード生成、事実に基づく質疑応答）には `temperature` を低く、創造的なタスク（ストーリー、ブレインストーミング）には `temperature` を相対的に高く設定するのが一般的な慣習です。

次、最終章ではこの本全体をまとめ、さらにその先へ進みたい読者のための学習パスを提案します。


# おわりに：全体のまとめと次のステップ

## 私たちが作ったものを振り返る

この本では、テキストの一行が実際にどのようなプロセスを経て次の単語を予測するのか、以下のパイプラインをモジュール単位で一つずつ直接実装してきました。```
mini_gpt/
├── tokenizer.py          # 第2章: テキスト -> トークンID (BPE)
├── embedding.py           # 第3章: トークンID -> ベクトル (トークン埋め込み + 位置埋め込み)
├── attention.py            # 第4章: Self-Attention, Multi-Head Attention
├── transformer_block.py    # 第5章: Attention + FeedForward + LayerNorm + Residual
├── model.py                 # 第6章: ブロックN個を積み重ねたGPTモデル全体
├── dataset.py                # 第7章: テキストを (入力, 正解) ペアに変換
├── train.py                   # 第7章: 学習ループ
└── generate.py                 # 第8章: テキスト生成とサンプリング戦略
```

そして第9章から11章では、この原理がGPT-2のような事前学習モデルにも同様に適用されること、LoRAによって軽量にファインチューニングできること、そして重みを変えずにプロンプトだけでモデルの振る舞いを制御できることを確認しました。

## 核心概念の要約

- **トークン化 (Tokenization)**: テキストをモデルが処理可能な整数の単位に分割する。BPEは頻出する断片を徐々に大きな単位へと結合し、語彙辞書を構築する。
- **埋め込み (Embedding)**: トークンIDを意味を持つベクトルに変換し、順序情報が失われないよう位置エンコーディング（Positional Embedding）を加える。
- **アテンション (Attention)**: Query-Key-Value構造を用い、各トークンが文脈内の他のトークンをどれだけ参照するかを計算する。Causal maskによって未来のトークンを隠すことで、「次単語予測」というタスクを成立させる。
- **Transformerブロック**: 残差接続（Residual Connection）とLayerNormで学習を安定させ、アテンション（トークン間の関係）とFeedForward（トークンごとの変換）を繰り返して表現力を高める。
- **学習 (Training)**: 元のテキストを1ステップずらして正解とする自己教師あり学習方式であり、別途の人手によるラベル付けなしに大量のデータを活用できる。
- **生成 (Generation)**: 学習された確率分布から、temperature、top-k、top-pを用いて次トークンを選択する手法により、出力結果の性質が変化する。
- **ファインチューニングとLoRA**: 全体の再学習を行わず、小さなアダプター行列のみを学習することで、少ないリソースでモデルを特定の目的に合わせて調整できる。
- **プロンプトエンジニアリング (Prompt Engineering)**: 重みを変えることなく、入力の設計のみによってモデルの出力を誘導する。

## 実用的な大規模LLMとの違い、さらに学ぶべきテーマ

本書のMini GPTは、原理を学ぶために意図的に縮小したモデルです。実際のプロダクションレベルのLLMへと進むには、以下のトピックを追加で調査することをお勧めします。

- **効率的なアテンション**: Flash AttentionやGrouped-Query Attention (GQA) など、長い文脈をより高速かつメモリ効率よく処理する技術。
- **位置エンコーディングの発展**: RoPE (Rotary Position Embedding) など、より長い文脈長への汎化性能に優れた位置エンコーディング方式。
- **RLHFとアライメント (Alignment)**: 人間のフィードバックを活用した強化学習により、モデルが有用かつ安全に応答するように調整するプロセス。
- **量子化 (Quantization)**: モデルの重みをより少ないビット数で表現し、メモリ使用量と演算量を削減する技術 (INT8, INT4など)。
- **MoE (Mixture of Experts)**: 入力ごとに全パラメータのうち一部のみを活性化させ、計算量に対してより大きなモデル容量を実現する構造。
- **RAG (Retrieval-Augmented Generation)**: 外部知識ベースを検索し、モデルの回答にリアルタイムな情報を結合する技術。
- **評価 (Evaluation)**: ベンチマークデータセットを用いて、モデルの推論、知識、安全性などを定量的に測定する手法。

## 推奨される学習パス

1. 本書の `MiniGPT` において、`embed_dim`、`num_layers`、`num_heads` を徐々に大きくしていき、パラメータ数と学習時間、生成品質の関係を直接観察してください。
2. 自作の日本語（または韓国語）コーパス（ニュース、ブログ、小説など）を用いて、トークナイザーとモデルを再学習させてみてください。
3. Hugging Faceの `Trainer` APIを用いて、フルファインチューニングとLoRAファインチューニングの結果を比較してください。
4. オリジナルの論文「Attention Is All You Need」とGPT-2の論文（"Language Models are Unsupervised Multitask Learners"）を読み、本書で実装したコードと論文内の数式を一つずつ照らし合わせてみてください。

## おわりに

LLMは魔法のように見えますが、本書で確認した通り、その中身はトークン化、行列演算、softmax、勾配降下法といった、よく定義された構成要素の集まりに過ぎません。規模が大きくなることで驚異的な能力が現れますが、その基礎は、皆さんが本書を通じてコードで直接触れたこの構造そのものです。この土台の上に立ち、より深い論文やオープンソースのコードを読み進めていけば、高度な発展技術もはるかにスムーズに理解できるようになるはずです。


# 用語集

本書で扱う核心的な人工知能およびLLM（大規模言語モデル）技術に関連する主要な用語の定義です。

* **BPE (Byte Pair Encoding)**: データ内で最も頻繁に出現するバイトペアを繰り返し結合することで、トークン辞書を構築するサブワード（Subword）ベースのトークナイザー・アルゴリズムです。OOV（Out-Of-Vocabulary）問題を解決します。
* **埋め込み (Embedding)**: 自然言語の単語やトークンを、コンピュータが理解可能な高次元の実数ベクトル空間上の座標にマッピングする技術です。単語の意味的な類似性を反映します。
* **Attention (アテンション)**: 入力されたシーケンスの中で、モデルがどのトークンに注目（重みを付与）すべきかを動的に決定するメカニズムです。
* **Self-Attention (セルフ・アテンション)**: 同一のシーケンス内のトークン同士で互いに重みを計算し、各トークンが文脈の中で持つ相互関係と意味を数値化する方式です。
* **Multi-Head Attention (マルチヘッド・アテンション)**: アテンションを複数の「ヘッド（Head）」に分割し、異なる表現空間（Representation Subspace）で並列に情報を処理した後に結合する構造です。
* **LayerNorm (レイヤー正規化)**: 各データサンプル内の特徴量（Feature）の平均と分散に基づき、活性化値を正規化する手法です。学習を安定させ、速度を向上させます。
* **Residual Connection (残差接続)**: レイヤーの入力を出力にそのまま加算するショートカット（Skip Connection）構造です。勾配消失（Vanishing Gradient）問題を大幅に緩和し、深いネットワークの学習を可能にします。
* **Transformer Block (トランスフォーマー・ブロック)**: マルチヘッド・アテンション、レイヤー正規化、残差接続、およびフィードフォワード・ニューラルネットワーク（FFN）で構成される、トランスフォーマー・アーキテクチャの基本単位構造です。
* **Decoder (デコーダー)**: 前のシーケンス情報（過去の予測トークン）に基づき、次のトークンを逐次的に予測（Auto-regressive）するためにマスキング処理されたSelf-Attention構造を含む、トランスフォーマーの一部です。GPT系モデルの骨格となります。
* **Sampling Strategy (サンプリング戦略)**: 生成モデルが次のトークンを選択する際、ランダム性と多様性を制御する方法です。
  - **Temperature**: ソフトマックス分布を調整し、テキストの創造性と決定論的な性質を支配するハイパーパラメータです。
  - **Top-K**: 確率の高い順に上位K個のトークンのみを候補群として絞り込み、サンプリングを行う手法です。
  - **Top-P (Nucleus Sampling)**: 累積確率分布がP以下となる最小限の候補群のみを選び出し、その中からサンプリングを行う手法です。
* **LoRA (Low-Rank Adaptation)**: 重み更新行列を2つの小さなランク（Low-Rank）行列に分解することで、学習パラメータ数を99%以上削減する効率的な微調整（PEFT）手法です。
